In [32]:
from __future__ import annotations
import asyncio
from typing import Optional, List
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi
import math
import pandas as pd
from runtime_support import (
    setup_client_from_env,
    api_navigate_ship,
    api_get_ship_nav,
    api_purchase_cargo,
    build_fleet_object
    )
from core_helpers import (
    init_world_state
)

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)

#Build all in-memory objects once (fleet_activity_obj, waypoints_ref_obj, waypoint_traits_obj, HQ)
world_state = await init_world_state(fleet_api, agents_api, systems_api)

In [34]:
from adapters.ships_activity_adapter import merge_activity_with_nav
from adapters.ships_specs_adapter import adapt_ships_specs_from_ship
from core_helpers import adapt_ships_activity_from_ship, upsert_many
from typing import Any, Dict, Iterable, List, Optional, Tuple
from runtime_support import call_sdk, unwrap_data
from dataclasses import dataclass, field
from domain.ships_activity import ShipsActivity
from domain.ships_specs import ShipsSpecs
from domain.ship_market import ShipMarketRow
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi

with setup_client_from_env() as client:
        fleet_api = FleetApi(client)
        agents_api = AgentsApi(client)
        systems_api = SystemsApi(client)

@dataclass
class FleetState:
    """Local, easily-referenced state for your session."""
    # Activity & Specs keyed by ship symbol
    activities: Dict[str, ShipsActivity] = field(default_factory=dict)
    specs: Dict[str, ShipsSpecs] = field(default_factory=dict)

    # Shipyard listings cached by waypoint
    ship_market: Dict[str, List[ShipMarketRow]] = field(default_factory=dict)

    def ensure_activity(self, symbol: str) -> ShipsActivity:
        a = self.activities.get(symbol)
        if a is None:
            a = ShipsActivity(symbol=symbol)
            self.activities[symbol] = a
        return a

    def update_activity_from_nav(self, symbol: str, nav_dto: Any) -> ShipsActivity:
        current = self.ensure_activity(symbol)
        updated = merge_activity_with_nav(current, nav_dto)
        self.activities[symbol] = updated
        return updated

    def update_activity_from_refuel(self, symbol: str, resp_dto: Any) -> ShipsActivity:
        """Use refuel response to update fuel state in local activity."""
        current = self.ensure_activity(symbol)
        fuel_cur = getattr(getattr(resp_dto, "fuel", None), "current", None)
        fuel_cap = getattr(getattr(resp_dto, "fuel", None), "capacity", None)
        patched = current.model_copy(update={
            "fuel_current": fuel_cur if fuel_cur is not None else current.fuel_current,
            "fuel_capacity": fuel_cap if fuel_cap is not None else current.fuel_capacity,
        })
        self.activities[symbol] = patched
        return patched

async def local_fleet_state(fleet: FleetApi) -> FleetState:
    """Load ships once, build local state (activity + specs) and persist to DB."""
    resp = await call_sdk(fleet, "get_my_ships")
    ships: Iterable[Any] = unwrap_data(resp)
    ships = list(ships)
    if not ships:
        raise SystemExit("[FATAL] No ships returned; check token/agent.")

    activities = [adapt_ships_activity_from_ship(d) for d in ships]
    specs = [adapt_ships_specs_from_ship(d) for d in ships]
    state = FleetState(
        activities={a.symbol: a for a in activities if a.symbol},
        specs={s.symbol: s for s in specs if s.symbol},
    )
    return state

# --------- define ship roles -----------
state = await local_fleet_state(fleet_api)
print(state.specs)

# Extract symbol for a given role
def get_symbol_by_role(ships_dict, role):
    for ship in ships_dict.values():
        if ship.role == role:
            return ship.symbol
    return None

command_ship = get_symbol_by_role(state.specs, "COMMAND")
sattelite = get_symbol_by_role(state.specs, "SATELLITE")
print(command_ship)

{'KIJINIBIBI-1': ShipsSpecs(symbol='KIJINIBIBI-1', role='COMMAND', frame_name='Frigate', frame_module_slots=8, frame_mounting_points=5, engine_name='Ion Drive II', speed=36, mounts=['MOUNT_SENSOR_ARRAY_II', 'MOUNT_GAS_SIPHON_II', 'MOUNT_MINING_LASER_II', 'MOUNT_SURVEYOR_II'], modules=['MODULE_CARGO_HOLD_II', 'MODULE_CREW_QUARTERS_I', 'MODULE_CREW_QUARTERS_I', 'MODULE_MINERAL_PROCESSOR_I', 'MODULE_GAS_PROCESSOR_I'], capacity=40), 'KIJINIBIBI-2': ShipsSpecs(symbol='KIJINIBIBI-2', role='SATELLITE', frame_name='Probe', frame_module_slots=0, frame_mounting_points=0, engine_name='Impulse Drive I', speed=9, mounts=[], modules=[], capacity=0)}
KIJINIBIBI-1


In [40]:
wps = world_state.waypoints.by_symbol
print(wps.keys())
dest = wps['X1-SV25-B20'].symbol


dict_keys(['X1-SV25-A1', 'X1-SV25-CX5E', 'X1-SV25-B6', 'X1-SV25-B7', 'X1-SV25-B8', 'X1-SV25-B9', 'X1-SV25-B10', 'X1-SV25-B11', 'X1-SV25-B12', 'X1-SV25-B13', 'X1-SV25-B14', 'X1-SV25-B15', 'X1-SV25-B16', 'X1-SV25-B17', 'X1-SV25-B18', 'X1-SV25-B19', 'X1-SV25-B20', 'X1-SV25-B21', 'X1-SV25-B22', 'X1-SV25-B23', 'X1-SV25-B24', 'X1-SV25-B25', 'X1-SV25-B26', 'X1-SV25-B27', 'X1-SV25-B28', 'X1-SV25-B29', 'X1-SV25-B30', 'X1-SV25-B31', 'X1-SV25-B32', 'X1-SV25-B33', 'X1-SV25-B34', 'X1-SV25-B35', 'X1-SV25-B36', 'X1-SV25-B37', 'X1-SV25-B38', 'X1-SV25-C39', 'X1-SV25-C41', 'X1-SV25-D42', 'X1-SV25-E44', 'X1-SV25-F46', 'X1-SV25-G51', 'X1-SV25-H52', 'X1-SV25-I56', 'X1-SV25-I57', 'X1-SV25-J58', 'X1-SV25-J59', 'X1-SV25-J60', 'X1-SV25-J61', 'X1-SV25-J62', 'X1-SV25-J63', 'X1-SV25-J64', 'X1-SV25-J65', 'X1-SV25-J66', 'X1-SV25-J67', 'X1-SV25-J68', 'X1-SV25-J69', 'X1-SV25-J70', 'X1-SV25-J71', 'X1-SV25-J72', 'X1-SV25-J73', 'X1-SV25-J74', 'X1-SV25-J75', 'X1-SV25-J76', 'X1-SV25-J77', 'X1-SV25-J78', 'X1-SV25-J79', 'X1

In [ ]:
print(nav_resp)

In [41]:
nav_resp = await api_navigate_ship(fleet_api, sattelite, dest)

starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Prep complete
KIJINIBIBI-2  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 654.95058
Arrived and ready
